In [ ]:
import requests
import json
import csv
import time
import logging
from tqdm import tqdm
import pandas as pd
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def requests_retry_session(
    retries=3,
    backoff_factor=0.3,
    session=None,
):
    session = session or requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def search_wikipedia_topics(search_term, min_words, max_words, headers, max_articles):
    base_url = "https://hi.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": search_term,
        "srnamespace": "0",
        "srlimit": "1000",  # Maximum limit per query
        "format": "json"
    }
    
    articles = []
    sroffset = 0  # Offset for pagination
    
    session = requests_retry_session()
    
    while len(articles) < max_articles:
        params['sroffset'] = sroffset
        try:
            response = session.get(base_url, params=params, headers=headers)
            response.raise_for_status()
            data = response.json()
            
            results = data.get('query', {}).get('search', [])
            if not results:
                break  # No more results to fetch
            
            for result in tqdm(results, desc=f"Searching articles for {search_term}"):
                title = result['title']
                article = fetch_wikipedia_article(title, min_words, max_words, headers, search_term)
                if article:
                    articles.append(article)
                
                time.sleep(0.1)  # Delay to respect rate limits
                
                if len(articles) >= max_articles:
                    break
            
            sroffset += len(results)  # Move to the next batch of results
            
        except requests.exceptions.RequestException as e:
            logging.error(f"Error searching for topics with term {search_term}: {e}")
            break

    return articles

def fetch_wikipedia_article(title, min_words, max_words, headers, search_term):
    base_url = "https://hi.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "exintro": "",
        "explaintext": "",
        "format": "json"
    }
    
    try:
        response = requests.get(base_url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        
        for page in data['query']['pages'].values():
            if 'extract' in page:
                content = page['extract']
                words = len(content.split())
                if min_words <= words <= max_words:  # Check if word count is between 400 and 800
                    logging.info(f"Successfully fetched article: {title} with {words} words")
                    return {
                        'title': page['title'],
                        'content': content,
                        'word_count': words,
                        'topic': search_term.split()[0]  # Adjust if Hindi topics have spaces
                    }
                else:
                    logging.info(f"Article '{title}' has {words} words, outside range {min_words}-{max_words}")
            else:
                logging.warning(f"No extract found for {title}")
    except requests.exceptions.RequestException as e:
        logging.error(f"Error fetching {title}: {e}")
    return None

def main():
    articles = []
    max_articles = 3000
    min_words = 400  # New minimum word count
    max_words = 800  # New maximum word count
    
    headers = {
        'User-Agent': 'MultiTopicArticlesFetcher/1.0 (yourname@example.com)'
    }
    
    # List of 60 Hindi topics
    topics = [
        "शिक्षा का महत्व"
        "पर्यावरण संरक्षण"
        "प्रदूषण की समस्या"
        "तकनीक का विकास"
        "मोबाइल फोन के फायदे और नुकसान"
        "स्वच्छ भारत अभियान"
        "जल संरक्षण"
        "स्वास्थ्य और स्वच्छता"
        "योग का महत्व"
        "भारतीय संस्कृति"
        "महिलाओं का सशक्तिकरण"
        "बेरोजगारी की समस्या"
        "ग्लोबल वार्मिंग"
        "सोशल मीडिया का प्रभाव"
        "ग्रामीण विकास"
        "शहरीकरण के प्रभाव"
        "खेलों का महत्व"
        "भारतीय त्योहार"
        "समय प्रबंधन"
        "किताबों का महत्व"
        "इंटरनेट का उपयोग"
        "विज्ञान के चमत्कार"
        "बाल मजदूरी"
        "भ्रष्टाचार की समस्या"
        "गरीबी उन्मूलन"
        "सड़क सुरक्षा"
        "भारतीय अर्थव्यवस्था"
        "जैविक खेती"
        "ऊर्जा संरक्षण"
        "जनसंख्या वृद्धि"
        "आत्मनिर्भर भारत"
        "कला और संस्कृति"
        "मानसिक स्वास्थ्य"
        "खाद्य सुरक्षा"
        "भारतीय रेलवे"
        "प्राकृतिक आपदाएँ"
        "स्वतंत्रता संग्राम"
        "डिजिटल भारत"
        "कन्या भ्रूण हत्या"
        "वन संरक्षण"
        "भारतीय संविधान"
        "अंतरिक्ष अनुसंधान"
        "कृत्रिम बुद्धिमत्ता"
        "पर्यटन का महत्व"
        "भारतीय सिनेमा"
        "जलवायु परिवर्तन"
        "स्वस्थ जीवन शैली"
        "शिक्षा में तकनीक का उपयोग"
        "भारतीय कृषि"
        "नारी शिक्षा"
        "युवाओं की भूमिका"
        "पशु संरक्षण"
        "सौर ऊर्जा"
        "अंतरराष्ट्रीय संबंध"
        "हिंदी भाषा का महत्व"
        "वृद्धावस्था की देखभाल"
        "मेक इन इंडिया"
        "सामाजिक समानता"
        "नशा मुक्ति"
        "लोकतंत्र का महत्व"
    ]
    
    for topic in topics:
        if len(articles) < max_articles:
            articles.extend(search_wikipedia_topics(topic, min_words, max_words, headers, max_articles - len(articles)))
        if len(articles) >= max_articles:
            break
    
    # Save to CSV
    csv_file = 'multi_topic_articles_hindi_400_800_2.csv'
    with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['title', 'content', 'word_count', 'topic']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for article in articles[:max_articles]:
            writer.writerow(article)
    
    # Convert CSV to Excel
    df = pd.read_csv(csv_file, encoding='utf-8')
    df.to_excel('multi_topic_articles_hindi_400_800_2.xlsx', index=False)
    logging.info("CSV file has been converted to Excel format.")

if __name__ == "__main__":
    main()

SyntaxError: unterminated string literal (detected at line 124) (3103674520.py, line 124)